# DataLoader

Wraps a `Dataset` (map-style or iterable) and handles **batching, shuffling, and parallel loading**. It's what you actually iterate over in the training loop.

### 1. Structure

```python
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,        # only for map-style Dataset
    num_workers=4,       # parallel processes loading data
    pin_memory=True,     # faster CPU -> GPU transfer
    drop_last=True,      # drop the last incomplete batch
)

for x, y in loader:      # each iteration gives 1 batch
    ...
```

### 2. What it does

- **Batching:** groups samples into batches (adds a new first dimension).
- **Shuffling:** shuffles the indices each epoch (`shuffle=True`).
- **Parallel loading:** `num_workers` processes prepare batches while the GPU trains.
- **Collating:** stacks samples into tensors (customizable with `collate_fn`).

### 3. Main parameters

| Parameter | What it does |
|---|---|
| `batch_size` | Samples per batch |
| `shuffle` | Shuffle every epoch (map-style only) |
| `num_workers` | Number of loading processes (0 = main process) |
| `pin_memory` | Speeds up transfer to GPU |
| `drop_last` | Drops the last batch if smaller than `batch_size` |
| `collate_fn` | Custom way to merge samples into a batch (e.g. padding) |

### 4. With each Dataset type

| | `Dataset` (map-style) | `IterableDataset` |
|---|---|---|
| `shuffle=True` | works | error |
| `num_workers > 0` | splits indices, no duplicates | duplicates unless you split shards (`get_worker_info()`) |
| `len(loader)` | works | not available |

### Tips

- Start with `num_workers=2-4` and tune from there.
- Use `pin_memory=True` when training on GPU.
- Don't set `shuffle=True` with `IterableDataset`; shuffle shards or use a buffer instead.

In [4]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self):
        self.data = [10, 20, 30, 40, 50, 60]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


dataset = MyDataset()

loader = DataLoader(
    dataset,
    batch_size = 2,
    shuffle = True,
)

In [6]:
for batch in loader:
    print(batch)

tensor([20, 10])
tensor([30, 50])
tensor([60, 40])
